# 🧠 Chain of Thought Prompting

**Welcome!** This notebook is your hands-on introduction to one of the most powerful ideas in prompt engineering: **Chain of Thought (CoT)** reasoning.

You'll discover that a single sentence added to a prompt can turn a confused, error-prone model into a careful, step-by-step problem solver. Pretty wild, right?

---

**How to use this notebook:**
- Cells marked **[RUN]** — just execute them, no changes needed.
- Cells marked **[TODO]** — you need to fill in some code before running.

> ⚠️ **Important:** Please select the **TORCH** kernel before starting.

## 1. Setup the environment and define utility functions

**[RUN]** The cells below install dependencies and load the model. No edits needed — just run them!

In [ ]:
#@title Install Dependencies {display-mode: "form"}
#@markdown Run this cell first to install the required packages.
!pip install transformers accelerate

In [ ]:
#@title Load the Qwen Model {display-mode: "form"}
#@markdown Loads a small Qwen model for fast GPU inference.
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.eval()

In [ ]:
#@title Define Model Response Utility {display-mode: "form"}
#@markdown Helper function that calls the model and returns the response text along with the elapsed time.
import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib


def generate_response(prompt):
    """
    Runs the model on a given prompt and returns the response text and elapsed time.
    """
    start = time.time()
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    response_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    elapsed = time.time() - start
    return str(response_text), elapsed

In [ ]:
#@title Define Display Utilities {display-mode: "form"}
#@markdown Styled HTML helpers for rendering prompts and model responses in a formatted table.
def display_sample(question, answer):
    display(HTML(
        f"""<p>Question/answer 0:</p>
        <strong>Question: </strong>{question}
        <p><strong>Answer: </strong>{answer}</p>
        """
    ))

def display_qwen(prompt):
    """
    Displays the prompt and model response in a styled table,
    with a loading indicator while the model generates text.
    """
    uid = str(uuid.uuid4()).replace("-", "")

    # HTML scaffold (before generation)
    html = f"""
    <style>
    .prompt-table {{
        border-collapse: collapse;
        width: 100%;
        margin: 12px 0;
        font-family: 'Segoe UI', sans-serif;
    }}
    .prompt-table th, .prompt-table td {{
        border: 1px solid #ccc;
        padding: 10px;
        vertical-align: top;
    }}
    .prompt-table th {{
        background-color: #f2f2f2;
        width: 20%;
    }}
    .loading {{
        color: #888;
        font-style: italic;
        animation: pulse 1.5s infinite;
    }}
    @keyframes pulse {{
        0% {{ opacity: 0.3; }}
        50% {{ opacity: 1; }}
        100% {{ opacity: 0.3; }}
    }}
    </style>
    <table class="prompt-table">
        <tr><th>Prompt</th><td>{prompt}</td></tr>
        <tr><th>Model Response</th><td id="response_{uid}">
            <span class="loading">⏳ Generating response...</span>
        </td></tr>
    </table>
    """

    # Display initial table
    display(HTML(html))

    # --- Call the model separately ---
    response_text, elapsed = generate_response(prompt)
    # print(response_text)
    # Escape special chars for HTML display
    safe_response = html_lib.escape(response_text.strip())

    # --- Inject response dynamically ---
    js = Javascript(f"""
        document.getElementById("response_{uid}").innerHTML =
            `<pre style="white-space: pre-wrap;">{safe_response}</pre>
             <div style='color:#666; font-size:90%; margin-top:4px;'>⏱ Generated in {elapsed:.2f} seconds</div>`;
    """)
    display(js)

## 2. Chain of Thought (CoT) Prompting

### What's the big idea?

Imagine asking someone a hard math question. If you just say *"Answer this"*, they might guess. But if you say *"Think through it step by step"*, suddenly they slow down, reason carefully, and are much more likely to get it right.

**Chain of Thought prompting does exactly this for LLMs.** By nudging the model to reason explicitly before answering, we unlock dramatically better performance on complex tasks — especially math, logic, and multi-step problems.

We'll test this on **GSM8K**, a popular benchmark of grade-school math word problems. Spoiler: the difference will be very noticeable! 🚀

**[RUN]** Let's start by loading the GSM8K dataset and peeking at a sample question and its answer.

In [ ]:
train_split_main = load_dataset("openai/gsm8k", "main", split="train[:1]")
print(f"Train split size: {len(train_split_main)}")

In [ ]:
display_sample(train_split_main[0]['question'], train_split_main[0]['answer'])

**[RUN]** Now let's ask our model to solve this question — no guidance, no hints, just the raw question (**zero-shot prompting**).

In [ ]:
question = train_split_main[0]['question']
prompt = f"""{question}"""

display_qwen(prompt)

Hm, not great 😬. The model jumped straight to an answer without really thinking things through. This is what *pattern matching without reasoning* looks like.

### ✏️ [TODO] Your turn — add a CoT trigger!

The fix is surprisingly simple. Complete the `cot_prompt` variable below with a short phrase that encourages the model to **think step by step** before answering.

> 💡 **Hint:** Think about how you'd ask a student to slow down and show their work. Even 5–6 words can be enough!

In [ ]:
cot_prompt = "Let's solve this problem step-by-step."
prompt = f"""{cot_prompt}
{question}"""

display_qwen(prompt)

🎉 **Amazing!** Such a tiny addition made such a big difference — the model is now reasoning step by step instead of just guessing.

This is the essence of Chain of Thought prompting: **words shape thinking**, even for AI systems.